In [0]:
customers_df = (
            spark.readStream
                        .format("cloudFiles")
                        .option("cloudFiles.format",'json')
                        .option("cloudFiles.schemaLocation", "/Volumes/gizmobox/landing/operational_data/cutomers_autoloader/_schema")
                        .option("cloudFiles.inferCoulumnTypes", "true")
                        .option("cloudFiles.schemaHints", "date_of_birth date, member_since date, created_timestamp timestamp")
                        .load("/Volumes/gizmobox/landing/operational_data/customers_autoloader/")
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

customers_transformed_df = (
    customers_df
        .withColumn("file_path", col("_metadata.file_path"))
        .withColumn("ingestion_date", current_timestamp())
)

In [0]:
streaming_query = (
    customers_transformed_df.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/customers_autoloader/_checkpoints/")
    .toTable("gizmobox.bronze.customers_autoloader")
)

In [0]:
streaming_query.stop()

In [0]:
%sql
select * from gizmobox.bronze.customers_autoloader